# Grid method (winner)

In [1]:
import numpy as np
import matplotlib.pyplot as plt
from scipy.signal import correlate, correlation_lags
from pathlib import Path
import napari

# ==========================================
# 1. CONSENSUS-BASED TFM STITCHER CLASS
# ==========================================
class TFMStitcher:
    def __init__(self, vol1, vol2, axis=2):
        self.vol1 = vol1.astype(np.float32)
        self.vol2 = vol2.astype(np.float32)
        self.axis = axis
        self.shift_index = 0
        self.best_score = -1

    def find_optimal_shift(self, grid=(120, 40), ignore_top=30, expected=0, tolerance=200, cutoff_db=-10):
        # --- GLOBAL dB CUTOFF LOGIC ---
        # Find the global peak between these two specific volumes
        v1_abs = np.abs(self.vol1)
        v2_abs = np.abs(self.vol2)
        global_max = max(np.max(v1_abs), np.max(v2_abs))
        
        # Calculate physical threshold
        thresh = global_max * (10**(cutoff_db / 20))
        
        # Apply threshold (silence everything below the gate)
        v1_t = np.where(v1_abs >= thresh, self.vol1, 0)
        v2_t = np.where(v2_abs >= thresh, self.vol2, 0)

        z_dim, y_dim, x_dim = v1_t.shape
        z_start, z_end = ignore_top, z_dim
        
        tile_z = (z_end - z_start) // grid[0]
        tile_y = y_dim // grid[1]
        
        all_shifts = []
        all_weights = []
        
        for r in range(grid[0]):
            for c in range(grid[1]):
                zs, ze = z_start + (r * tile_z), z_start + ((r + 1) * tile_z)
                ys, ye = c * tile_y, (c + 1) * tile_y
                
                # Use the thresholded data for profile extraction
                prof1 = np.max(np.abs(v1_t[zs:ze, ys:ye, :]), axis=(0, 1))
                prof2 = np.max(np.abs(v2_t[zs:ze, ys:ye, :]), axis=(0, 1))

                # Skip tiles that are entirely silenced by the cutoff
                if np.max(prof1) == 0 or np.max(prof2) == 0:
                    continue
                
                # ZNCC Normalization
                p1_n = (prof1 - np.mean(prof1)) / (np.std(prof1) + 1e-10)
                p2_n = (prof2 - np.mean(prof2)) / (np.std(prof2) + 1e-10)
                
                corr = correlate(p1_n, p2_n, mode='full')
                lags = correlation_lags(len(p1_n), len(p2_n), mode='full')
                
                # Constraints (Directional + Expected range)
                mask = (lags >= expected - tolerance) & (lags <= expected + tolerance)
                mask = mask & (lags >= 0) 
                if not np.any(mask): continue
                corr[~mask] = -np.inf 

                peak_idx = np.argmax(corr)
                # Recency bias (weight tiles at the edge of the assembly more)
                dist_from_edge = c / grid[1] 
                bias = 0.5 + (0.5 * dist_from_edge) 
                
                all_shifts.append(lags[peak_idx])
                all_weights.append(corr[peak_idx] * bias)

        if not all_shifts:
            print(f"  !! Warning: No features survived {cutoff_db}dB cutoff for this pair.")
            self.shift_index = 0
            return 0

        # Weighted Consensus
        lag_min, lag_max = np.min(all_shifts), np.max(all_shifts)
        bins = np.arange(lag_min, lag_max + 2) - 0.5
        counts, bin_edges = np.histogram(all_shifts, bins=bins, weights=all_weights)
        
        self.shift_index = int(bin_edges[np.argmax(counts)] + 0.5)
        self.best_score = np.max(counts)
        return self.shift_index

    def stitch(self, blend_mode='linear'):
        shift = int(round(self.shift_index))
        s1, s2 = self.vol1.shape, self.vol2.shape
        L1, L2 = s1[self.axis], s2[self.axis]
        
        v1_start, v2_start = (abs(shift), 0) if shift < 0 else (0, shift)
        total_len = max(v1_start + L1, v2_start + L2)
        
        stitched = np.zeros((*s1[:self.axis], total_len, *s1[self.axis+1:]), dtype=np.float32)
        
        sl1, sl2 = [slice(None)]*3, [slice(None)]*3
        sl1[self.axis], sl2[self.axis] = slice(v1_start, v1_start+L1), slice(v2_start, v2_start+L2)
        stitched[tuple(sl1)] = self.vol1
        
        inter_s, inter_e = max(v1_start, v2_start), min(v1_start + L1, v2_start + L2)
        if inter_e > inter_s:
            overlap = inter_e - inter_s
            ramp_shape = [1, 1, 1]; ramp_shape[self.axis] = overlap
            ramp = np.linspace(0, 1, overlap).reshape(ramp_shape)
            if shift < 0: ramp = 1.0 - ramp
            
            idx1, idx2 = [slice(None)]*3, [slice(None)]*3
            idx1[self.axis], idx2[self.axis] = slice(inter_s-v1_start, inter_e-v1_start), slice(inter_s-v2_start, inter_e-v2_start)
            stitched[tuple(idx1)] = (self.vol1[tuple(idx1)] * (1-ramp)) + (self.vol2[tuple(idx2)] * ramp)

        if v2_start + L2 > inter_e:
            end_sl = [slice(None)]*3
            end_sl[self.axis] = slice(inter_e, v2_start+L2)
            src_sl = [slice(None)]*3
            src_sl[self.axis] = slice(inter_e-v2_start, L2)
            stitched[tuple(end_sl)] = self.vol2[tuple(src_sl)]
            
        return stitched

# ==========================================
# 2. MAIN SEQUENTIAL EXECUTION
# ==========================================
if __name__ == "__main__":
    IN_DIR = Path.cwd().parent / 'DATA' / '2D TFM Data' / 'FeC Smile 3MHz 04022026 Filtered'

    selected = [
        IN_DIR / "FeC_40_9_filtered_3D_TFM.npy",
        IN_DIR / "FeC_40_8_filtered_3D_TFM.npy",
        IN_DIR / "FeC_40_7_filtered_3D_TFM.npy",
        IN_DIR / "FeC_40_6_filtered_3D_TFM.npy",
        IN_DIR / "FeC_40_5_filtered_3D_TFM.npy",
        IN_DIR / "FeC_40_4_filtered_3D_TFM.npy",
        IN_DIR / "FeC_40_3_filtered_3D_TFM.npy",
        IN_DIR / "FeC_40_2_filtered_3D_TFM.npy",
        IN_DIR / "FeC_40_1_filtered_3D_TFM.npy"
    ]
    
    print(f"Found {len(selected)} volumes. Starting Sequential Cutoff-Stitch...")
    global_vol = np.load(selected[0])
    
    for i in range(1, len(selected)):
        next_vol = np.load(selected[i])
        print(f"\nMerging {i+1}/{len(selected)}: {selected[i].name}")

        # Tail-Slicing: Only compare the new image to the end of the current assembly
        window_size = int(next_vol.shape[2] * 1.2)
        offset_removed = max(0, global_vol.shape[2] - window_size)
        active_tail = global_vol[:, :, offset_removed:]

        # Find shift using the -10dB cutoff logic inside the class
        stitcher = TFMStitcher(active_tail, next_vol, axis=2)
        relative_shift = stitcher.find_optimal_shift(grid=(120, 40), ignore_top=30, cutoff_db=-10)
        
        # Translate to global coordinate space
        global_shift = offset_removed + relative_shift
        
        print(f" -> Consensus Shift: {global_shift} px (Local: {relative_shift}) | Conf: {stitcher.best_score:.2f}")
        
        # Perform the actual linear-blend stitch on the full global volume
        full_stitcher = TFMStitcher(global_vol, next_vol, axis=2)
        full_stitcher.shift_index = global_shift
        global_vol = full_stitcher.stitch()
        
    # --- ROBUST VISUALIZATION ---
    # Using sorted percentiles to prevent monotonicity errors
    v_min = float(np.percentile(global_vol, 0.1))
    v_max = float(np.percentile(global_vol, 99.9))
    clim = sorted([v_min, v_max])
    if clim[0] == clim[1]: clim = [clim[0], clim[0] + 1]
    
    viewer = napari.Viewer(title="Cutoff Sequential Assembly")
    viewer.add_image(
        global_vol, 
        name="Final Assembly", 
        contrast_limits=clim, 
        colormap="viridis"
    )

    napari.run()

Found 9 volumes. Starting Sequential Cutoff-Stitch...

Merging 2/9: FeC_40_8_filtered_3D_TFM.npy
 -> Consensus Shift: 107 px (Local: 107) | Conf: 3115.96

Merging 3/9: FeC_40_7_filtered_3D_TFM.npy
 -> Consensus Shift: 197 px (Local: 130) | Conf: 2973.56

Merging 4/9: FeC_40_6_filtered_3D_TFM.npy
 -> Consensus Shift: 296 px (Local: 139) | Conf: 3437.48

Merging 5/9: FeC_40_5_filtered_3D_TFM.npy
 -> Consensus Shift: 353 px (Local: 97) | Conf: 1435.77

Merging 6/9: FeC_40_4_filtered_3D_TFM.npy
 -> Consensus Shift: 471 px (Local: 158) | Conf: 1191.56

Merging 7/9: FeC_40_3_filtered_3D_TFM.npy
 -> Consensus Shift: 592 px (Local: 161) | Conf: 2030.82

Merging 8/9: FeC_40_2_filtered_3D_TFM.npy
 -> Consensus Shift: 707 px (Local: 155) | Conf: 2104.50

Merging 9/9: FeC_40_1_filtered_3D_TFM.npy
 -> Consensus Shift: 800 px (Local: 133) | Conf: 4210.39
